In [1]:
import pandas as pd
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from glob import glob
torch.manual_seed(42)
np.random.seed(42)
import copy

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!unzip -q "/content/drive/MyDrive/dataset1.zip" -d "/content/"

In [5]:
path = "/content/dataset"

In [6]:
classes = {'no': 0, 'sphere': 1, 'vort': 2}

print("--- Inspecting .npy file properties (Train Split) ---")

# Point it specifically to the 'train' subfolder
split_dir = os.path.join(path, 'train')

for name in classes.keys():
    folder = os.path.join(split_dir, name)
    files = sorted(glob(os.path.join(folder, '*.npy')))

    # Safety check to avoid IndexError if a folder is empty
    if not files:
        print(f"\n⚠️ WARNING: No .npy files found in: {folder}")
        continue

    sample_file = files[0]
    data = np.load(sample_file)

    print(f"\nClass: '{name}'")
    print(f"File: {os.path.basename(sample_file)}")
    print(f"  Shape:     {data.shape}")
    print(f"  Data Type: {data.dtype}")
    print(f"  Min Value: {data.min():.4f}")
    print(f"  Max Value: {data.max():.4f}")
    print(f"  Mean:      {data.mean():.4f}")
    print(f"  Std Dev:   {data.std():.4f}")

--- Inspecting .npy file properties (Train Split) ---

Class: 'no'
File: 1.npy
  Shape:     (1, 150, 150)
  Data Type: float64
  Min Value: 0.0000
  Max Value: 1.0000
  Mean:      0.0635
  Std Dev:   0.1024

Class: 'sphere'
File: 1.npy
  Shape:     (1, 150, 150)
  Data Type: float64
  Min Value: 0.0000
  Max Value: 1.0000
  Mean:      0.0582
  Std Dev:   0.0958

Class: 'vort'
File: 1.npy
  Shape:     (1, 150, 150)
  Data Type: float64
  Min Value: 0.0000
  Max Value: 1.0000
  Mean:      0.0519
  Std Dev:   0.0918


In [7]:
def load_split(path, split):
    paths, labels = [], []
    split_dir = os.path.join(path, split)
    for name, label in classes.items():
        folder = os.path.join(split_dir, name)
        files = sorted(glob(os.path.join(folder, '*.npy')))
        paths.extend(files)
        labels.extend([label] * len(files))
        print(f'  [{split}] {name}: {len(files)} samples')
    return paths, labels

train_paths, train_labels = load_split(path, 'train')
val_paths,   val_labels   = load_split(path, 'val')

print(f'\nTotal Train: {len(train_paths)} | Total Val: {len(val_labels)}')

  [train] no: 10000 samples
  [train] sphere: 10000 samples
  [train] vort: 10000 samples
  [val] no: 2500 samples
  [val] sphere: 2500 samples
  [val] vort: 2500 samples

Total Train: 30000 | Total Val: 7500


In [8]:
class dataset(Dataset):
    def __init__(self,paths,labels,transform=None):
        self.paths= paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)
    def  __getitem__(self , ind):
        img = torch.from_numpy(np.load(self.paths[ind]).astype(np.float32))
        if self.transform:
            image = self.transform(img)
        return img, torch.tensor(self.labels[ind], dtype=torch.long)


transformer = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
])

train_loader= DataLoader(dataset(train_paths, train_labels, transformer),
                          batch_size=64, shuffle=True)

test_loader= DataLoader(dataset(val_paths, val_labels),
                          batch_size=64, shuffle=False)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

Train batches: 469 | Test batches: 118


In [9]:
#efficientNetB0
def efficientnet(n=3):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    old_cnn = model.features[0][0]
    model.features[0][0] = nn.Conv2d(
        1, old_cnn.out_channels,
        kernel_size=old_cnn.kernel_size,
        stride=old_cnn.stride,
        padding=old_cnn.padding,
        bias=False
    )
    with torch.no_grad():
        model.features[0][0].weight =nn.Parameter(old_cnn.weight.mean(dim=1,keepdim=True) )

    in_features =model.classifier[1].in_features
    model.classifier =nn.Sequential( nn.Dropout(0.4),nn.Linear(in_features,n))
    return model

effnet =efficientnet().to(device)



Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 152MB/s]


In [10]:
def each_epoch(model, loader, optimizer , criterion ):
    model.train()
    total_loss, correct ,total  = 0.0,0.0,0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1)== labels).sum().item()
        total += imgs.size(0)

    return total_loss/total , correct /total

@torch.no_grad()
def evaluate(model ,loader ,criterion):
    model.eval()
    total_loss, correct,total = 0.0,0.0,0
    probs, y_labels = [],[]
    for imgs,labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss= criterion(outputs,labels)
        total_loss +=loss.item()*imgs.size(0)
        correct+= (outputs.argmax(1)==labels).sum().item()
        total+=imgs.size(0)
        probs.append(torch.softmax(outputs, dim=1).cpu().numpy())
        y_labels.append(labels.cpu().numpy())

    return total_loss / total, correct / total, np.concatenate(probs), np.concatenate(y_labels)

def train_model(model , model_name, train_loader=train_loader, test_loader=test_loader , epochs=25, lr= 1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc, best_weights = 0.0, None

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = each_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc ,_ , _ =evaluate(model,train_loader,criterion)
        scheduler.step()

        for key, val in zip(history.keys(), [tr_loss, tr_acc, va_loss, va_acc]):
            history[key].append(val)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_weights = copy.deepcopy(model.state_dict())

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}/{epochs} | '
                  f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
                  f'Val Loss: {va_loss:.4f} Acc: {va_acc:.4f}')

    model.load_state_dict(best_weights)
    print(f'\n Best Val Accuracy: {best_val_acc:.4f}')
    return history

In [11]:
history_eff = train_model(effnet, 'EfficientNet-B0', epochs=25, lr=3e-4)

torch.save(effnet.state_dict(), 'efficientnet_best.pth')

  Epoch   1/25 | Train Loss: 0.9892 Acc: 0.4754 | Val Loss: 0.7463 Acc: 0.6586
  Epoch   5/25 | Train Loss: 0.3360 Acc: 0.8672 | Val Loss: 0.2653 Acc: 0.8894
  Epoch  10/25 | Train Loss: 0.1166 Acc: 0.9560 | Val Loss: 0.0646 Acc: 0.9761
  Epoch  15/25 | Train Loss: 0.0429 Acc: 0.9841 | Val Loss: 0.0052 Acc: 0.9987
  Epoch  20/25 | Train Loss: 0.0238 Acc: 0.9905 | Val Loss: 0.0004 Acc: 1.0000
  Epoch  25/25 | Train Loss: 0.0174 Acc: 0.9933 | Val Loss: 0.0001 Acc: 1.0000

 Best Val Accuracy: 1.0000
